# This is your first ChatBot

### It is an extension of previous Practicals when you could query a .pdf document, except that now you can have an entire conversation about the document. It uses `LangChain` as a development environmennt. This first Notebook uses a previous version of LangChain, which is good to start with (and still works). In the next Notebook, you will see a more recent version. This first exercise is about porting a previous apporoach to a dialogue context, and getting started with Prompt writing for ChatBots.

In [2]:
%%capture
!pip install python-dotenv
!pip install openai
!pip install langchain==0.1.0
!pip install pdfplumber

import os
import openai
import pdfplumber

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from langchain.chat_models import ChatOpenAI
from langchain import OpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.memory import ConversationBufferWindowMemory
from langchain.schema import SystemMessage
from langchain.chains import LLMChain
from langchain.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    StringPromptTemplate,
    MessagesPlaceholder,
    BaseChatPromptTemplate
)

## Import a .pdf file whose contents you want to have a dialogue about. Keep it shorter than 20 pages.

In [7]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

  # Now you can work with the uploaded file, e.g., using pdfplumber:
  with pdfplumber.open(fn) as pdf:
    for page in pdf.pages:
      my_doc = page.extract_text()

Saving CountryProfile_Pakistan.pdf to CountryProfile_Pakistan (1).pdf
User uploaded file "CountryProfile_Pakistan (1).pdf" with length 3876038 bytes


## You need to author a Prompt that will govern how the Bot behaves. This is the main learning exercise.

In [8]:
prompt = PromptTemplate(
    template="""
You are a highly knowledgeable travel guide specializing in Pakistan. Your role is to help users explore Pakistan by providing detailed, insightful, and engaging information about its history, culture, geography, economy, and tourist attractions. Every response must be based solely on the reference document provided below.
---
IMPORTANT:
Your reference document for answering questions is provided in this file:
<REF>
{my_doc}
</REF>
---
Response Guidelines:
---
Response Guidelines:
- **Word Count:**
  Each response must be between 50 and 200 words.

- **Tone & Style:**
  - Use warm, friendly, and respectful language.
  - Adjust your tone based on the user's mood. If the user sounds excited, serious, or emphatic, match that tone with a more energetic, formal, or light style accordingly.

- **Content Accuracy & Protection of Pakistan's Image:**
  - Base all your details on the provided reference document.
  - Ensure your responses accurately represent Pakistan’s rich culture, history, and natural beauty.
  - If any negative or misleading information arises—especially from external media—calm the user by emphasizing that media reports can be biased or inaccurate. For example, you might say, "Don't believe everything you see in the media; let me share the true experience of Pakistan."
  - If certain details are missing from the reference, mention that limitation and offer general, reliable travel advice.

- **Structure & Formatting:**
  - Begin with a brief greeting or introduction.
  - Organize the main content with clear headings and bullet points.
  - For topics like history, start with a clear section header (e.g., **History:**) followed by your explanation.
  - Leave a blank line between sections for better readability.
  - Conclude your answer with a follow-up question or an invitation for further discussion.

- **Engagement & Clarification:**
  - If the user's query is vague or broad, ask clarifying questions such as:
      - "Could you please specify which aspect you are most interested in?"
      - "Would you like more details on historical events, cultural practices, or geographical features?"
      - "Is there a particular region in Pakistan you want to explore?"
  - **If the user responds with 'yes' to a follow-up question:**
      - Acknowledge their affirmative response and provide additional details directly.

- **Personalization & Encouragement:**
  - Tailor your examples and recommendations directly to the user's expressed interests.
  - Encourage the user to ask further questions or share feedback to create an interactive experience.

- **Evaluation Awareness:**
  - Understand that your responses will be compared against a consistent set of questions.
  - Aim to be clear, accurate, and engaging every time, and continuously improve your responses based on feedback.

---
Current Conversation History:
{history}

User Query:
{human_input}

Bot Response:
""",
    input_variables=["my_doc", "history", "human_input"]
)

prompt_formatted_str = prompt.format(
    my_doc=my_doc,
    history="{history}",
    human_input="{human_input}"
)

prompt = PromptTemplate(
    input_variables=["history", "human_input"],
    template=prompt_formatted_str
)


In [ ]:
key = "API KEY"

### This is the ChatBot core. You are creating a Chain which contains:
*   an LLM (here gpt-4o-mini)
*   your Prompt
*   a memory buffer so that the Bot remembers previous turns in the conversation




In [10]:
llm = ChatOpenAI(openai_api_key=key, model="gpt-4o-mini", temperature=0.7)

memory = ConversationBufferWindowMemory(
    memory_key="history", k=7, return_messages=True)

chat_llm_chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    verbose=False)

/usr/local/lib/python3.11/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


### A minimal User Interface to run the ChatBot. It also logs the dialogue into a file (NOTE: writing to the file is "a" mode).

In [11]:
import ipywidgets as widgets
from IPython.display import display
import textwrap

# Create UI elements
user_input_box = widgets.Text(
    placeholder="Enter your message here...",
    description="User:",
    style={'description_width': 'initial'}
)
chat_output = widgets.Output()
end_chat_button = widgets.Button(description="End Chat", button_style="danger")

# Display UI elements
display(chat_output, user_input_box, end_chat_button)

# Open a file for saving dialogue
Dfile = open("dialogue_log.txt", "a")

def handle_input(text):
    user_input = text.value.strip()

    if not user_input:
        return  # Ignore empty input

    if user_input.lower() in ["exit", "quit", "fin", "bye", "fini"]:
        with chat_output:
            print("Chat ended by user.")
        user_input_box.disabled = True
        return

    # Write user input to file
    print("User: " + user_input, file=Dfile)

    # Process response (replace with actual call to OpenAI API)
    response = chat_llm_chain.predict(human_input=user_input)

    # Wrap text for better display
    wrapped_response = textwrap.fill(response, width=120)

    with chat_output:
        print(f"User: {user_input}")
        print(f"Bot: {wrapped_response}")

    # Write bot response to file
    print("Bot: " + response, file=Dfile)

    # Clear input box for next message
    user_input_box.value = ""

def stop_chat(_):
    with chat_output:
        print("Chat ended by user.")
    user_input_box.disabled = True
    Dfile.close()

# Event bindings
user_input_box.on_submit(handle_input)  # Waits for Enter before triggering
end_chat_button.on_click(stop_chat)


Output()

Text(value='', description='User:', placeholder='Enter your message here...', style=DescriptionStyle(descripti…

Button(button_style='danger', description='End Chat', style=ButtonStyle())

# Step 1 in Bot development is to understand the various elements, and how the ChatBot behaviour depends on its Prompt. Developing an ChatBot involves:
*   Choosing how it accesses specific information
*   Setting up a dialogue context (which is a loop within which completion is applied to each user input)
*   Authoring a Prompt governing dialogue behaviour